In [1]:
# ========================================
# Medium Training: 1 Shard, 10 Epochs
# ========================================

import sys
sys.path.append('../')

import torch
from torch.utils.data import DataLoader, random_split
from src.dataset import NTIRETrainDataset, get_transforms
from src.models import CLIPDetector
from src.train import train_model

import warnings
warnings.filterwarnings('ignore')

import os
os.environ['PYTHONWARNINGS'] = 'ignore'

print("=" * 60)
print("MEDIUM TRAINING")
print("=" * 60)

# Config
BATCH_SIZE = 16
EPOCHS = 10
LR = 1e-4
ACCUMULATION_STEPS = 2  # Effective batch = 32
VAL_SPLIT = 0.1

# Device
device = torch.device("mps" if torch.backends.mps.is_available() else "cpu")
print(f"Device: {device}")

# Load dataset (1 shard = 50k images)
print("\nLoading dataset (shard 0)...")
train_transform, val_transform = get_transforms()

full_dataset = NTIRETrainDataset(
    '../data/train', 
    shard_nums=[0],  # Only shard 0
    transform=train_transform
)

# Train/Val split
val_size = int(len(full_dataset) * VAL_SPLIT)
train_size = len(full_dataset) - val_size

train_dataset, val_dataset = random_split(
    full_dataset,
    [train_size, val_size],
    generator=torch.Generator().manual_seed(42)
)

print(f"Train: {len(train_dataset)} | Val: {len(val_dataset)}")

# DataLoaders
train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=2,
    pin_memory=False
)

val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=2,
    pin_memory=False
)

# Model
print("\nCreating model...")
model = CLIPDetector()

# Train
print("\nStarting training...")
print(f"  Epochs: {EPOCHS}")
print(f"  Batch size: {BATCH_SIZE} (effective: {BATCH_SIZE * ACCUMULATION_STEPS})")
print(f"  Learning rate: {LR}")
print(f"  Train samples: {len(train_dataset)}")
print(f"  Val samples: {len(val_dataset)}")
print("=" * 60)

model, history = train_model(
    model,
    train_loader,
    val_loader,
    epochs=EPOCHS,
    lr=LR,
    device=device,
    save_path='../checkpoints/clip_medium.pth',
    accumulation_steps=ACCUMULATION_STEPS
)

print("\n✅ Medium training completed!")
print(f"Best model saved at: checkpoints/clip_medium.pth")

/Library/Frameworks/Python.framework/Versions/3.13/lib/python3.13/site-packages/albumentations/check_version.py:147: UserWarning: Error fetching version info <urlopen error [SSL: CERTIFICATE_VERIFY_FAILED] certificate verify failed: unable to get local issuer certificate (_ssl.c:1018)>
  data = fetch_version_info()


MEDIUM TRAINING
Device: mps

Loading dataset (shard 0)...
Loaded 1 shards: 50000 images
  Real: 17982 | Fake: 32018
Train: 45000 | Val: 5000

Creating model...
Loading openai/clip-vit-base-patch16...


Loading weights:   0%|          | 0/398 [00:00<?, ?it/s]

CLIPModel LOAD REPORT from: openai/clip-vit-base-patch16
Key                                  | Status     |  | 
-------------------------------------+------------+--+-
vision_model.embeddings.position_ids | UNEXPECTED |  | 
text_model.embeddings.position_ids   | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


✓ Backbone frozen (only classifier trainable)
✓ Parameters: 197,121 trainable / 149,817,858 total

Starting training...
  Epochs: 10
  Batch size: 16 (effective: 32)
  Learning rate: 0.0001
  Train samples: 45000
  Val samples: 5000
Training started

Epoch 1/10
------------------------------------------------------------


...

Train Loss: 0.2690 | Val AUC: 0.9452
✓ Best model saved (AUC: 0.9452)

Epoch 9/10
------------------------------------------------------------


Validation: 100%|██████████| 313/313 [04:02<00:00,  1.29it/s]


Train Loss: 0.2666 | Val AUC: 0.9455
✓ Best model saved (AUC: 0.9455)

Epoch 10/10
------------------------------------------------------------


Validation: 100%|██████████| 313/313 [04:03<00:00,  1.29it/s]


Train Loss: 0.2643 | Val AUC: 0.9457
✓ Best model saved (AUC: 0.9457)
✓ Training curves saved: ../checkpoints/clip_medium_curves.png

Training completed! Best AUC: 0.9457

✅ Medium training completed!
Best model saved at: checkpoints/clip_medium.pth
